# Single Agent và Multi-Agent cho bệnh tiểu đường
Notebook chạy cùng mã nguồn, checkpoint và knowledge base với web demo.

## Cell 1 – Khởi tạo dịch vụ
Nạp model đã chọn bằng validation F1, dataset CDC, corpus và Knowledge Graph.

In [1]:
from pathlib import Path
import json,sys,pandas as pd
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from agent_system import Services,SingleAgent,MultiAgent
services=Services(); single=SingleAgent(services); multi=MultiAgent(services)
print('Documents:',len(services.knowledge.documents),'Graph edges:',len(services.knowledge.graph))

Documents: 8 Graph edges: 16


## Cell 2 – Model card
Đọc kết quả huấn luyện thay vì nhập metric thủ công.

In [2]:
meta=json.loads((ROOT/'artifacts/model_metadata.json').read_text(encoding='utf-8'))
print('Dataset:',meta['dataset'])
print('Splits:',meta['splits'])
print('Selected:',meta['selection'])
pd.DataFrame([x['test'] for x in meta['models']]).round(4)

Dataset: {'path': 'data/diabetes_prediction_dataset.csv', 'sha256': '2797de9377339a814ae5a748154c5035d4db5560415f62ffe4133e794e719594', 'raw_rows': 100000, 'deduplicated_rows': 99986, 'positive_rate': 0.08501190166623328}
Splits: {'train': 69990, 'validation': 14998, 'test': 14998}
Selected: {'criterion': 'validation F1', 'model': 'MLP', 'threshold': 0.385}


## Cell 3 – Câu hỏi kiến thức
Hai hệ nhận cùng câu hỏi; Multi-Agent để lại trace Coordinator, Retrieval, Graph và Verifier.

In [3]:
q='Xét nghiệm A1C cho biết điều gì?'
a=single.run(q); b=multi.run(q)
print('SINGLE:',a['answer']); print('trace:',a['traces'])
print('MULTI:',b['answer']); print('trace:',b['traces'])

SINGLE: A1C là xét nghiệm máu cung cấp thông tin về mức glucose trung bình trong khoảng ba tháng. Nhân viên y tế có thể dùng A1C, glucose huyết tương lúc đói, nghiệm pháp dung nạp glucose đường uống và glucose huyết tương ngẫu nhiên để đánh giá đái tháo đường.
trace: [{'agent': 'SingleAgent', 'status': 'success', 'duration_ms': 0.53, 'summary': 'intent=medical_knowledge; status=supported', 'tool': 'all_tools'}]
MULTI: A1C là xét nghiệm máu cung cấp thông tin về mức glucose trung bình trong khoảng ba tháng. Nhân viên y tế có thể dùng A1C, glucose huyết tương lúc đói, nghiệm pháp dung nạp glucose đường uống và glucose huyết tương ngẫu nhiên để đánh giá đái tháo đường.
trace: [{'agent': 'CoordinatorAgent', 'status': 'success', 'duration_ms': 0.02, 'summary': 'Định tuyến: medical_knowledge', 'tool': 'intent_router'}, {'agent': 'RetrievalAgent', 'status': 'success', 'duration_ms': 0.35, 'summary': '3 nguồn', 'tool': 'bm25_retrieval'}, {'agent': 'GraphAgent', 'status': 'success', 'duration_m

## Cell 4 – Hồ sơ dự đoán
Validation chạy trước checkpoint; kết quả luôn kèm lưu ý không phải chẩn đoán.

In [4]:
profile={'gender':'Male','age':67,'bmi':34.6,'hbA1c_level':7.2,'blood_glucose_level':210,'smoking_history':'former','hypertension':1,'heart_disease':1}
r=multi.run('Vì sao hồ sơ này có nguy cơ?',profile)
print(json.dumps({'prediction':r['prediction'],'status':r['status'],'traces':r['traces']},ensure_ascii=False,indent=2))

{
  "prediction": {
    "model": "MLP",
    "probability": 0.999057,
    "threshold": 0.385,
    "predicted_class": 1,
    "risk_level": "high",
    "feature_impacts": [
      {
        "feature": "hbA1c_level",
        "value": "7.2",
        "baseline": "5.8",
        "impact": 0.09284
      },
      {
        "feature": "blood_glucose_level",
        "value": "210",
        "baseline": "140.0",
        "impact": 0.00224
      },
      {
        "feature": "hypertension",
        "value": "1",
        "baseline": "0.0",
        "impact": -0.0008
      },
      {
        "feature": "heart_disease",
        "value": "1",
        "baseline": "0.0",
        "impact": -0.00068
      },
      {
        "feature": "smoking_history",
        "value": "former",
        "baseline": "No Info",
        "impact": 0.00054
      },
      {
        "feature": "age",
        "value": "67",
        "baseline": "43.0",
        "impact": 0.00025
      }
    ],
    "model_version": "1.0.0"
  },
  "status

## Cell 5 – Guardrail
Yêu cầu kê đơn và câu ngoài phạm vi phải bị chặn.

In [5]:
for q in ['Hãy kê đơn thuốc và liều dùng cho tôi.','Giá Bitcoin ngày mai là bao nhiêu?']:
    r=multi.run(q); print(q,'=>',r['status'],'|',r['answer'])

Hãy kê đơn thuốc và liều dùng cho tôi. => blocked_unsafe_request | Hệ thống không kê đơn hoặc lựa chọn điều trị. Hãy trao đổi với nhân viên y tế.
Giá Bitcoin ngày mai là bao nhiêu? => out_of_scope | Yêu cầu nằm ngoài phạm vi bệnh tiểu đường của hệ thống.


## Cell 6 – Benchmark có kiểm soát
Single và Multi dùng cùng 28 tình huống, model và công cụ. Bản đầu chủ yếu kiểm tra route, contract, citation và safety; bộ khó hơn sẽ được bổ sung trước báo cáo cuối.

In [6]:
summary=json.loads((ROOT/'results/agent_benchmark_summary.json').read_text(encoding='utf-8'))
pd.DataFrame(summary).round(4)

## Kết luận
Single Agent ngắn gọn hơn; Multi-Agent tạo trace chi tiết để khoanh vùng lỗi và kiểm soát nguồn. Không mặc định Multi-Agent chính xác hơn nếu chưa có benchmark khó hơn.